# **Method (01) - ImageDataGenerator()**
- Older Keras method
- Used for:
  - augmentation (rotation, zoom, flip)
  - loading images from folders
- Mostly legacy now
- cant split data into train, val only, so if need test dataset also we have to use another tool **split_folders**
   - run this command in cmd >>> *split_folders --output processed_data --ratio .7 .1 .2 -- data*
   - processed_data is the out put path and data is the input path 

In [37]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [38]:
# Before training, dataset is usually split into train/validation/test sets (e.g., 70/20/10). This can be done manually or using tools like split-folders.

# defining transformation rules
datagen_rules = ImageDataGenerator(
    rescale= 1./255,        # normalizing pixel values to 0-1 range
    horizontal_flip=True,   # randomly flips some images left <-> right
    rotation_range=10       # Randomly rotates images between -10° and +10°
)

# data generators, returns a Python iterator (DirectoryIterator) that yields batches

train_generator = datagen_rules.flow_from_directory(
    "split_data/train",
    target_size= (256, 256), # pixels
    batch_size=32,
    class_mode="sparse",
    save_to_dir="Datagen_result"
)

Found 1506 images belonging to 3 classes.


In [8]:
test_generator = datagen_rules.flow_from_directory(
    "split_data/test",
    target_size=(256, 256),
    batch_size=32,
    class_mode="sparse",
)

Found 431 images belonging to 3 classes.


In [9]:
val_generator = datagen_rules.flow_from_directory(
    "split_data/val",
    target_size=(256, 256),
    batch_size=32,
    class_mode="sparse"
)

Found 215 images belonging to 3 classes.


In [17]:
for img_batch,label_batch in train_generator:
    batch = img_batch
    print(img_batch[0])
    break

[[[0.66084415 0.653001   0.70790297]
  [0.6588898  0.65104663 0.7059486 ]
  [0.6569354  0.64909226 0.7039942 ]
  ...
  [0.7226414  0.7226414  0.7697002 ]
  [0.72991765 0.72991765 0.77697647]
  [0.73333335 0.73333335 0.7803922 ]]

 [[0.65932935 0.6514862  0.70638824]
  [0.65981793 0.6519748  0.7068768 ]
  [0.6603066  0.6524634  0.7073654 ]
  ...
  [0.7236186  0.7236186  0.77067745]
  [0.7304062  0.7304062  0.77746505]
  [0.73333335 0.73333335 0.7803922 ]]

 [[0.665594   0.65775084 0.7126528 ]
  [0.66461676 0.6567736  0.7116756 ]
  [0.6636396  0.65579647 0.7106984 ]
  ...
  [0.7245958  0.7245958  0.7716546 ]
  [0.7308948  0.7308948  0.7779536 ]
  [0.73333335 0.73333335 0.7803922 ]]

 ...

 [[0.56092954 0.5398692  0.57167757]
  [0.5627948  0.54318696 0.57063794]
  [0.588167   0.56855917 0.59601015]
  ...
  [0.60151356 0.5936704  0.6368077 ]
  [0.601025   0.59318185 0.6363191 ]
  [0.6005364  0.59269327 0.6358305 ]]

 [[0.5648383  0.5442666  0.5746091 ]
  [0.5608404  0.5412326  0.56868356]


In [19]:
type(train_generator)

keras.src.legacy.preprocessing.image.DirectoryIterator

In [36]:
len(train_generator) # not of batches

48

In [34]:
for i in train_generator:
    print(type(i))
    print(i[0])
    break

<class 'tuple'>
[[[[0.6666078  0.65876466 0.7136667 ]
   [0.79075813 0.782915   0.83781695]
   [0.7467231  0.73888    0.79378194]
   ...
   [0.45129952 0.44345638 0.49835834]
   [0.45335007 0.44550693 0.5004089 ]
   [0.45601368 0.4481705  0.5030725 ]]

  [[0.6645637  0.6567205  0.71162254]
   [0.7915015  0.7836584  0.83856034]
   [0.74585587 0.73801273 0.7929147 ]
   ...
   [0.61867845 0.6108353  0.6657373 ]
   [0.618734   0.61089087 0.6657928 ]
   [0.6177429  0.60989976 0.6648017 ]]

  [[0.66309065 0.6552475  0.71014947]
   [0.79143864 0.78359544 0.83849746]
   [0.74714535 0.7393022  0.7942042 ]
   ...
   [0.55607295 0.5482298  0.6031318 ]
   [0.5556178  0.5477747  0.60267663]
   [0.5541931  0.54634994 0.6012519 ]]

  ...

  [[0.65893966 0.64717495 0.7138416 ]
   [0.65887773 0.647113   0.7137797 ]
   [0.65890104 0.64713633 0.71380305]
   ...
   [0.4122903  0.38876086 0.44366285]
   [0.28138798 0.25785857 0.31276053]
   [0.57987833 0.5563489  0.61125094]]

  [[0.63218856 0.62042385 0.6

In [18]:
batch.shape

(32, 256, 256, 3)

### Note
- train_generator is jsut an obj, when we use it only do imaage processing things other wise nothing (train_generator is an iterator that lazily loads images, applies transformations, and yields batches during training)
- in train_generator can be iterated and returns tuples as (x_batch, y_batch)
- x_batch = numpy array of shape (batch_size, 256, 256, 3)
- y_batch = numpy array of shape (batch_size,) 
- each tuple contain two item, each item is a batch
- first batch in a tuple contain image data(32 images) as numbers, second batch contain labels(32) both are in type numpy array
- class_mode="sparse" means labels are integers (0,1,2,...) NOT one-hot encoding
- **We can directly pass train_generator into model.fit() because it is an iterator that yields batches of preprocessed and labeled images. However, we usually also specify steps_per_epoch to control how many batches make one epoch**

What does “generator is infinite” actually mean?
- internally it works like a loop:

while True:<br>
    load batch of images<br>
    apply augmentation<br>
    yield (x_batch, y_batch)<br>

- It just keeps generating data again and again
- not data is infinite iterator behavior is infinite
  - reuses them every epoch
  - shuffles them again
  - applies random augmentation again